# master UCM visualizacion avanzada
# Tema 7 shiny - Ejemplo Datos 4500 vehículos UK 2016

# Basado en parte en el ejemplo de Joe Cheng
# https://gist.github.com/jcheng5/3239667


In [1]:
import asyncio
import hypercorn.asyncio
import hypercorn.config
import pandas as pd
from shiny import *
from plotnine import *
import sklearn
import skmisc



In [2]:
# 1. Gestor para iniciar/detener la app sin bloquear Jupyter
app_tasks = []

class AppTask:
    def __init__(self, app: App, port: int = 8000):
        self.shutdown_event = asyncio.Event()
        self.app = app
        self.port = port
        self.task = asyncio.create_task(self.start())
        app_tasks.append(self)

    async def start(self):
        config = hypercorn.config.Config()
        config.bind = [f"localhost:{self.port}"]
        await hypercorn.asyncio.serve(
            self.app, config, shutdown_trigger=self.shutdown_event.wait
        )

    def stop(self):
        self.shutdown_event.set()

def stop_all_apps():
    for task in app_tasks:
        task.stop()
    app_tasks.clear()



In [3]:
# Cargamos el dataframe a partir del .csv (se hace antes de definir la app shiny *no dentro de la app*)

df = pd.read_csv("vehiculos_UK_2016.csv", sep = "|")
continuas = df.select_dtypes(include=['int', 'float']).columns.tolist()


In [4]:
df.head()

,Manufacturer,Model,EngineCapacity,FuelType,MetricUrbanCold,MetricExtraUrban,MetricCombined,ImperialUrbanCold,ImperialExtraUrban,ImperialCombined,CO2gkm,FuelCost12000Miles,EuroStandard,NoiseLeveldBA,EmissionsCOmgkm,THCEmissionsmgkm,EmissionsNOxmgkm,THC+NOxEmissionsmgkm,ParticulatesNo_mgkm,Tipo
0,ABARTH,"500, 2012 onwards",1368,Petrol,7.9,4.9,6.0,35.8,57.6,47.1,139,1170,6,72.5,455.0,23.0,9.0,NaN,NaN,Gasolina
1,ABARTH,"500, 2012 onwards",1368,Petrol,7.6,4.7,5.8,37.2,60.1,48.7,134,1131,6,73.0,445.0,25.0,22.0,NaN,NaN,Gasolina
2,ABARTH,"500, 2012 onwards",1368,Petrol,7.9,4.9,6.0,35.8,57.6,47.1,139,1170,6,74.0,222.0,35.0,19.0,NaN,NaN,Gasolina
3,ABARTH,"500, 2012 onwards",1368,Petrol,7.6,4.7,5.8,37.2,60.1,48.7,134,1131,6,73.5,439.0,34.0,37.0,NaN,NaN,Gasolina
4,ABARTH,"500, 2012 onwards",1368,Petrol,7.9,4.9,6.0,35.8,57.6,47.1,139,1170,6,71.5,498.0,36.0,27.0,NaN,NaN,Gasolina


In [5]:
continuas

['EngineCapacity',
 'MetricUrbanCold',
 'MetricExtraUrban',
 'MetricCombined',
 'ImperialUrbanCold',
 'ImperialExtraUrban',
 'ImperialCombined',
 'CO2gkm',
 'FuelCost12000Miles',
 'EuroStandard',
 'NoiseLeveldBA',
 'EmissionsCOmgkm',
 'THCEmissionsmgkm',
 'EmissionsNOxmgkm',
 'THC+NOxEmissionsmgkm',
 'ParticulatesNo_mgkm']

In [8]:
# 2. Define aquí tu aplicación Shiny (o impórtala desde tu archivo)


app_ui = ui.page_fluid(
    ui.layout_sidebar(
        ui.sidebar(
            ui.input_select('eje_x',
            'Elige variable para eje X', 
            continuas,
            selected = continuas[0]),
            ui.input_select('eje_y',
            'Elige variable para eje Y', 
            continuas,
            selected = continuas[1]),
        ),
        ui.card(
            ui.output_plot("plot"),
        ),
    ),
    )

def server(input, output, session):
    @output
    @render.plot(alt = "Gráfico Bivariante")
    def plot():
      p = ggplot(df, aes(x = str(input.eje_x()), 
                         y = str(input.eje_y()),
                         fill = "Tipo")) + geom_point() + geom_smooth(method = 'lowess') 
      return p

mi_app = App(app_ui, server)


In [9]:
# 3. Inicia la app en el puerto 8000
AppTask(mi_app, port=8000)

[2026-11-11 18:37:57 +0100] [61356] [INFO] Running on http://127.0.0.1:8000 (CTRL + C to quit)
/home/pedro/Escritorio/UCM_2026/entorno_julio/lib/python3.12/site-packages/plotnine/stats/smoothers.py:342: PlotnineWarning: Confidence intervals are not yet implemented for lowess smoothings.
/home/pedro/Escritorio/UCM_2026/entorno_julio/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_point : Removed 31 rows containing missing values.
/home/pedro/Escritorio/UCM_2026/entorno_julio/lib/python3.12/site-packages/plotnine/stats/smoothers.py:342: PlotnineWarning: Confidence intervals are not yet implemented for lowess smoothings.
/home/pedro/Escritorio/UCM_2026/entorno_julio/lib/python3.12/site-packages/plotnine/stats/smoothers.py:342: PlotnineWarning: Confidence intervals are not yet implemented for lowess smoothings.
/home/pedro/Escritorio/UCM_2026/entorno_julio/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_point : Removed 31 rows containing 

In [10]:
from IPython.display import IFrame

IFrame(src="http://localhost:8000", width="100%", height="400px")
